In [1]:
from pathlib import Path

from dataeval.config import set_seed
from maite_datasets.image_classification import MilitaryVehicles

# Seed the random state used by MiniBatchKMeans and KMeans clusterers
# so the sweep produces deterministic results across machines.
set_seed(42)

data_root = Path("./data")

# Download once; subsequent runs read existing disk files.
MilitaryVehicles(root=data_root, image_set="base", download=True)
MilitaryVehicles(root=data_root, image_set="train", as_datamaite=True)

# The export nests images under the split directory.
data_path = data_root / "militaryvehicles_datamaite_train" / "train"
print(f"Reading from {data_path}")

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading from data/militaryvehicles_datamaite_train/train


In [2]:
from dataeval_flow import PipelineConfig, run_task
from dataeval_flow.config import (
    HuggingFaceDatasetConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.config.extractors import BoVWExtractorConfig
from dataeval_flow.workflows.parameter_sweep import ParameterSweepConfig

# Define the sweep workflow
sweep_workflow = ParameterSweepConfig(
    name="mv_sensitivity_sweep",
    # Outlier parameters: sweep thresholds
    outlier_method=["adaptive"],
    outlier_threshold=[2.5, 3.5, 4.5],
    outlier_flags=["dimension", "pixel", "visual"],
    # Duplicate parameters: sweep sensitivity across documented 0.1-3.0 range
    duplicate_cluster_sensitivity=[0.5, 2.0, 3.0],
    duplicate_cluster_algorithm=["hdbscan"],
    duplicate_merge_near=True,
)

# Define the task referencing the sweep workflow
task = TaskConfig(name="mv_param_sweep", workflow="mv_sensitivity_sweep", sources="mv_src", extractor="bovw_ext")

# Build the full pipeline config
config = PipelineConfig(
    datasets=[
        HuggingFaceDatasetConfig(name="mv_train", path=str(data_path), task="image_classification"),
    ],
    views=[
        # Shuffle before limiting to avoid sampling only the first alphabetical classes.
        ViewConfig(
            name="sample1000",
            operations=[
                ViewOperation(type="Shuffle", params={"seed": 42}),
                ViewOperation(type="Limit", params={"size": 1000}),
            ],
        ),
    ],
    sources=[
        SourceConfig(name="mv_src", dataset="mv_train", view="sample1000"),
    ],
    extractors=[
        BoVWExtractorConfig(name="bovw_ext", vocab_size=256, batch_size=32),
    ],
    workflows=[sweep_workflow],
    tasks=[task],
)

In [3]:
result = run_task(task, config, cache_dir=Path("./cache"))

In [4]:
print(result.report())


  PARAMETER SWEEP COMPLETE. 9 COMBINATIONS EVALUATED.
  Timestamp:    2026-09-25T21:45:31.790818+00:00
  Duration:     0.80s
  Source:       mv_src (mv_train[sample1000])
  Model:        bovw_ext (bovw)
------------------------------------------------------------------------------------------

  SUMMARY
  -------
  Outliers Sweep ........................................... 3 unique combinations  [..]
  Near Duplicates Sweep .................................... 3 unique combinations  [..]

  Health: All checks passed [ok]

  OUTLIERS SWEEP                                                     3 unique combinations
  Effect of outlier_threshold on outliers.

  outlier_threshold  Outliers
  -----------------  --------
  2.5                     157
  3.5                     112
  4.5                      96

  NEAR DUPLICATES SWEEP                                              3 unique combinations
  Effect of duplicate_cluster_sensitivity on near duplicates.

  duplicate_cluster_sensitivity